<a href="https://colab.research.google.com/github/sibandze/Bird-Intelligence-System/blob/dev-unsupervised/notebooks/exploration_and_data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Notebook: Data Loading + Spectogram Pipeline + Exploration

---



**Mount Google Drive & Define Directories**

In [ ]:
import os
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Master Google Drive folder for dataset backups
DRIVE_BACKUP_DIR = '/content/drive/MyDrive/Bird-Intelligence-System_data_and_outputs'
os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
print(f"Drive backup directory ready: {DRIVE_BACKUP_DIR}")

**Clone Repository & Install Dependencies**

In [ ]:
import os
import sys

REPO_URL = "https://github.com/sibandze/Bird-Intelligence-System.git"
BRANCH = "dev-unsupervised"
REPO_DIR = "/content/Bird-Intelligence-System"

if not os.path.exists(REPO_DIR):
    print("Cloning repo...")
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print("Repo exists. Pulling latest...")
    %cd {REPO_DIR}
    !git fetch origin {BRANCH}
    !git reset --hard origin/{BRANCH}  # this overwrites local changes

%cd {REPO_DIR}

if os.path.exists("requirements.txt"):
    !pip install -r requirements.txt

Load Config & Restore Existing Files from Google Drive

In [ ]:
import re
import shutil
import yaml

CONFIG_PATH = os.path.join(REPO_DIR, "configs", "config.yaml")

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

# Extract relative paths from config
raw_audio_dir = config['data']['raw_audio_dir']             # data/raw_audio
processed_npy_dir = config['data']['processed_npy_dir']     # data/processed_spectrograms
metadata_dir = config['data']['metadata_dir']               # data/metadata

# Ensure target local directories exist inside the cloned repo
for rel_dir in [raw_audio_dir, processed_npy_dir, metadata_dir]:
    os.makedirs(os.path.join(REPO_DIR, rel_dir), exist_ok=True)

# Audio & Spectrogram Config Parameters to match against file naming schemes
audio_params = config.get('audio', {})
sr = audio_params.get('sr', 32000)
n_fft = audio_params.get('n_fft', 2048)
hop_length = audio_params.get('hop_length', 512)
n_mels = audio_params.get('n_mels', 128)

# Expected parameter string in processed file names (e.g., _sr32000_nfft2048_hop512_nmel128_)
spectrogram_pattern = f"sr{sr}_nfft{n_fft}_hop{hop_length}_nmel{n_mels}"

print(f"Scanning Drive backup using pattern key: '{spectrogram_pattern}'...\n")

def restore_files_from_drive(target_rel_dir, file_filter_fn):
    drive_dir = os.path.join(DRIVE_BACKUP_DIR, target_rel_dir)
    local_dir = os.path.join(REPO_DIR, target_rel_dir)

    restored_count = 0
    if os.path.exists(drive_dir):
        for filename in os.listdir(drive_dir):
            drive_file_path = os.path.join(drive_dir, filename)
            local_file_path = os.path.join(local_dir, filename)

            if os.path.isfile(drive_file_path) and file_filter_fn(filename):
                if not os.path.exists(local_file_path):
                    shutil.copy2(drive_file_path, local_file_path)
                    restored_count += 1

        print(f"[{target_rel_dir}] Restored {restored_count} matching file(s) from Drive.")
    else:
        print(f"[{target_rel_dir}] No Drive directory found at {drive_dir}")

# 1. Restore Raw Audio (.ogg / .wav files)
restore_files_from_drive(
    raw_audio_dir,
    lambda f: f.endswith(('.ogg', '.wav', '.mp3'))
)

# 2. Restore Spectrograms matching current audio parameters
restore_files_from_drive(
    processed_npy_dir,
    lambda f: f.endswith('.npy') and spectrogram_pattern in f
)

# 3. Restore Metadata files (.csv)
restore_files_from_drive(
    metadata_dir,
    lambda f: f.endswith('.csv') and spectrogram_pattern in f
)

# Optional: Override/update runtime config parameters
config['training']['batch_size'] = 32
config['training']['epochs'] = 50
config['logging']['use_wandb'] = False

RUN_CONFIG_PATH = os.path.join(REPO_DIR, "configs", "config_colab_run.yaml")
with open(RUN_CONFIG_PATH, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print("\nConfiguration resolved and Drive files restored to local workspace.")

In [ ]:
# Execute the pipeline runner module with the updated config
!python -m pipeline.pipeline_runner --config configs/config_colab_run.yaml

In [ ]:
TARGET_DATA_DIRS = [
    config['data']['metadata_dir'],
    config['data']['processed_npy_dir'],
    config['data']['raw_audio_dir']
]

for rel_dir in TARGET_DATA_DIRS:
    local_dir_path = os.path.join(REPO_DIR, rel_dir)
    drive_dir_path = os.path.join(DRIVE_BACKUP_DIR, rel_dir)

    if os.path.exists(local_dir_path):
        os.makedirs(drive_dir_path, exist_ok=True)

        files_copied = 0
        for filename in os.listdir(local_dir_path):
            src_file = os.path.join(local_dir_path, filename)
            dest_file = os.path.join(drive_dir_path, filename)

            if os.path.isfile(src_file) and not os.path.exists(dest_file):
                shutil.copy2(src_file, dest_file)
                files_copied += 1

        print(f"Synced {files_copied} new file(s) from '{rel_dir}' to Drive -> '{drive_dir_path}'")

print("\nBackup complete!")